# Data Understanding and checking null values and types

In [3]:
from pathlib import Path

project_root = Path.cwd().parent
data_dir = project_root / "data" / "raw"

In [4]:
import pandas as pd

# loading data_dictionary csv
data_dictionary = pd.read_csv(data_dir/ "data_dictionary.csv")

data_dictionary.info()

<class 'pandas.DataFrame'>
RangeIndex: 57 entries, 0 to 56
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   file         57 non-null     str  
 1   column       57 non-null     str  
 2   description  57 non-null     str  
 3   dtype        57 non-null     str  
dtypes: str(4)
memory usage: 1.9 KB


In [5]:
# loading sales file

sales = pd.read_csv(data_dir/ "bookstore_sales.csv")

print(sales.info())
print(sales.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 13152 entries, 0 to 13151
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           13152 non-null  str    
 1   year           13152 non-null  int64  
 2   month          13152 non-null  int64  
 3   store_id       13152 non-null  str    
 4   channel        13152 non-null  str    
 5   transactions   13152 non-null  int64  
 6   revenue        13152 non-null  float64
 7   gst_collected  13152 non-null  float64
 8   net_revenue    13152 non-null  float64
 9   dataset        13152 non-null  str    
dtypes: float64(3), int64(3), str(4)
memory usage: 1.0 MB
None
date             0
year             0
month            0
store_id         0
channel          0
transactions     0
revenue          0
gst_collected    0
net_revenue      0
dataset          0
dtype: int64


In [6]:
print(sales['date'].dtype)

str


In [7]:
# convertt date column from string to datetime object
sales['date'] = pd.to_datetime(sales['date'])

In [8]:
sales['date'].dtype

dtype('<M8[us]')

In [9]:
import pandas as pd
from pathlib import Path

files = list(data_dir.glob("*.csv"))

for f in files:
    df = pd.read_csv(f)
    print(f"--- {f.name} ---")
    print(f"shape: {df.shape}")
    print(df.dtypes)
    print("nulls:\n", df.isnull().sum())
    print()

--- Bookstore Checking Balanced Dataset.csv ---
shape: (1258, 7)
date                   str
description            str
deposit            float64
withdrawal         float64
account_id             str
net_change         float64
rolling_balance    float64
dtype: object
nulls:
 date               0
description        0
deposit            0
withdrawal         0
account_id         0
net_change         0
rolling_balance    0
dtype: int64

--- Bookstore Credit Balance Dataset.csv ---
shape: (978, 7)
date                   str
description            str
charge             float64
payment            float64
card_id                str
net_change         float64
rolling_balance    float64
dtype: object
nulls:
 date               0
description        0
charge             0
payment            0
card_id            0
net_change         0
rolling_balance    0
dtype: int64

--- bookstore_customers.csv ---
shape: (12000, 7)
customer_id        int64
signup_date          str
home_store           str
annua

In [10]:
dfs = {}
for f in files:
    df = pd.read_csv(f)


    # 1. Find all columns in this specific dataframe that contain 'date'
    date_columns = [col for col in df.columns if 'date' in col.lower()]

    print(date_columns)
    # 2. Loop through and convert only those identified columns
    for col in date_columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')
        
    dfs[f.stem] = df

['date']
['date']
['signup_date']
['signup_date', 'first_purchase_date', 'last_purchase_date']
['signup_date', 'first_purchase_date', 'last_purchase_date']
['hire_date', 'term_date']
[]
[]
['date']
[]


In [11]:
print(dfs['bookstore_employees'].dtypes)

employee_id                 int64
name                          str
location                      str
role                          str
employment_type               str
hourly_rate               float64
hire_date          datetime64[us]
term_date          datetime64[us]
dataset                       str
dtype: object


# Connections between files

In [16]:
print(dfs['bookstore_employees'].columns)
print(dfs['bookstore_payroll'].columns)

Index(['employee_id', 'name', 'location', 'role', 'employment_type',
       'hourly_rate', 'hire_date', 'term_date', 'dataset'],
      dtype='str')
Index(['pay_period_start', 'employee_id', 'employee_name', 'location', 'role',
       'employment_type', 'hourly_rate', 'hours_biweekly', 'gross_pay',
       'cpp_withheld', 'ei_withheld', 'income_tax_withheld',
       'employee_benefits', 'net_pay', 'dataset'],
      dtype='str')


In [17]:
merge_payroll = pd.merge(
    dfs['bookstore_employees'], 
    dfs['bookstore_payroll'], 
    on='employee_id', 
    validate='one_to_many'
).drop(columns=['location_x', 'role_x', 'employment_type_x', 'hourly_rate_x'])

In [20]:
print(len(merge_payroll))
print(len(dfs['bookstore_payroll']))

6640
6640


In [27]:
print(dfs['bookstore_sales'].columns)

Index(['date', 'year', 'month', 'store_id', 'channel', 'transactions',
       'revenue', 'gst_collected', 'net_revenue', 'dataset'],
      dtype='str')


In [26]:
# difference between customer and customer_expanded
customer = dfs['bookstore_customers']
customer_expanded = dfs['bookstore_customers_expanded']

print(len(customer), len(customer_expanded))
print(customer.columns)
print(customer_expanded.columns)

12000 12000
Index(['customer_id', 'signup_date', 'home_store', 'annual_visits',
       'online_ratio', 'ltv_estimate', 'dataset'],
      dtype='str')
Index(['customer_id', 'signup_date', 'home_store', 'annual_visits',
       'online_ratio', 'ltv_estimate', 'first_purchase_date',
       'last_purchase_date', 'region', 'loyalty_member', 'total_purchases',
       'total_spent', 'avg_purchase_value', 'visits_2023', 'has_escalation',
       'escalation_reason', 'customer_rating'],
      dtype='str')


In [28]:
print(customer_expanded.head())

   customer_id signup_date home_store  annual_visits  online_ratio  \
0            1  2019-06-08     YYC-NW             18          0.25   
1            2  2023-12-21     YYC-NW              9          0.47   
2            3  2021-03-10     ONLINE              3          0.25   
3            4  2021-08-13     ONLINE             15          0.43   
4            5  2024-02-12     YYC-NW              2          0.25   

   ltv_estimate first_purchase_date last_purchase_date      region  \
0         62.70          2019-06-08         2019-11-22  Calgary NW   
1         40.67          2023-12-21         2024-12-16  Calgary NW   
2        220.50          2021-03-10         2022-01-10      Online   
3        466.80          2021-08-13         2023-11-03      Online   
4        179.24          2024-02-12         2024-09-01  Calgary NW   

   loyalty_member  total_purchases  total_spent  avg_purchase_value  \
0           False               84      6117.54               72.83   
1           Fals

In [29]:
print(dfs['bookstore_sales'].head())

        date  year  month store_id   channel  transactions  revenue  \
0 2019-01-01  2019      1   YYC-DT  IN_STORE            64  1985.46   
1 2019-01-01  2019      1   YYC-DT    ONLINE             8   207.72   
2 2019-01-01  2019      1   YYC-NW  IN_STORE            49  1466.95   
3 2019-01-01  2019      1   YYC-NW    ONLINE             4   148.89   
4 2019-01-01  2019      1   YYC-SE  IN_STORE            33  1272.70   

   gst_collected  net_revenue          dataset  
0          99.27      1886.19  bookstore_sales  
1          10.39       197.33  bookstore_sales  
2          73.35      1393.60  bookstore_sales  
3           7.44       141.45  bookstore_sales  
4          63.64      1209.06  bookstore_sales  


In [ ]:
# checking why customers_expanded_raw has 12,966 as compare to customers_expanded 12,000
dfs['bookstore_customers_expanded_raw']['customer_id'].duplicated().sum()

np.int64(966)

In [30]:
dfs['bookstore_loans'].head()

,quarter_end,loan_name,loan_id,principal_outstanding,interest_accrued,repayment,dataset
0,2020-03-30,Government Relief Loan,GOVT-CEBA,40000.0,0.0,0.0,bookstore_loans
1,2020-06-30,Government Relief Loan,GOVT-CEBA,40000.0,0.0,0.0,bookstore_loans
2,2020-09-30,Government Relief Loan,GOVT-CEBA,40000.0,0.0,0.0,bookstore_loans
3,2020-12-30,Government Relief Loan,GOVT-CEBA,40000.0,0.0,0.0,bookstore_loans
4,2021-03-30,Government Relief Loan,GOVT-CEBA,40000.0,0.0,0.0,bookstore_loans


In [31]:
dfs['Bookstore Checking Balanced Dataset'].head()

,date,description,deposit,withdrawal,account_id,net_change,rolling_balance
0,2019-01-01,Sales sweep deposit,4726.44,0.00,CHK-001,4726.44,79726.44
1,2019-01-01,Store & warehouse rent,0.00,18500.00,CHK-001,-18500.00,61226.44
2,2019-01-02,Misc operating expense,0.00,486.05,CHK-001,-486.05,60740.39
3,2019-01-03,Payroll,0.00,11758.92,CHK-001,-11758.92,48981.47
4,2019-01-04,Sales sweep deposit,4102.04,0.00,CHK-001,4102.04,53083.51


In [32]:
dfs['Bookstore Credit Balance Dataset'].head()

,date,description,charge,payment,card_id,net_change,rolling_balance
0,2019-01-01,Software subscriptions,768.71,0.0,CC-001,768.71,768.71
1,2019-01-03,E-comm shipping labels,287.53,0.0,CC-001,287.53,1056.24
2,2019-01-07,E-comm shipping labels,188.86,0.0,CC-001,188.86,1245.10
3,2019-01-09,Office & packaging supplies,119.61,0.0,CC-001,119.61,1364.71
4,2019-01-16,Office & packaging supplies,376.30,0.0,CC-001,376.30,1741.01


In [33]:
print(dfs['Bookstore Credit Balance Dataset'].columns)
print(dfs['Bookstore Checking Balanced Dataset'].columns)
print(dfs['bookstore_loans'].columns)

Index(['date', 'description', 'charge', 'payment', 'card_id', 'net_change',
       'rolling_balance'],
      dtype='str')
Index(['date', 'description', 'deposit', 'withdrawal', 'account_id',
       'net_change', 'rolling_balance'],
      dtype='str')
Index(['quarter_end', 'loan_name', 'loan_id', 'principal_outstanding',
       'interest_accrued', 'repayment', 'dataset'],
      dtype='str')


In [35]:
print(dfs['Bookstore Checking Balanced Dataset']['account_id'].unique())
print(dfs['Bookstore Credit Balance Dataset']['card_id'].unique())

<StringArray>
['CHK-001']
Length: 1, dtype: str
<StringArray>
['CC-001']
Length: 1, dtype: str


### Data connections:
- we can find the payroll and revenue difference by connecting sales and payrolls employees are connected through employeee id.
- we can also find which branch is making more revenue by sales through store id
- we can figure out the cashflow using loans, sales, payrolls
- customers are connected to sales through store id and through time. 
- Checking (CHK-001) and credit (CC-001) are single whole-business accounts, not tied to individual stores. They connect to sales/payroll/loans only through time (monthly/quarterly alignment), no shared ID exists
- Loans connect the same way, only through quarter_end, no ID links it to any other file